# SPRINT 10

Jessica Beltran Rodriguez

# Sprint APIs — Consultar APIs REST amb `requests`

> **Enunciat**
>
> **Descripció**
>
> Aplicaràs el que has après sobre APIs REST, mètodes HTTP i codis d'estat, utilitzant APIs públiques per consultar dades, filtrar-les i finalment desar-les en un DataFrame de pandas.
>
> **Recursos:**
> - Nivell 1: [{JSON} Placeholder](https://jsonplaceholder.typicode.com/)
> - Nivell 2: [Try Public APIs for free](https://github.com/public-apis/public-apis)
> - Nivell 3: [Open Data BCN](https://opendata-ajuntament.barcelona.cat/ca/desenvolupadors#APIS)

---
Aplicació de mètodes HTTP (GET, POST, PATCH, DELETE), codis d'estat i conversió de respostes JSON a DataFrames de pandas.

In [1]:
# requests: llibreria per fer peticions HTTP (GET, POST, PATCH, DELETE...) des de Python
import requests
# pandas: llibreria per treballar amb taules de dades (DataFrames)
import pandas as pd

---
# Nivell 1 — Exploració bàsica amb JSONPlaceholder

> **Enunciat**
>
> Des d'un Jupyter Notebook faràs els següents exercicis utilitzant la llibreria requests de Python.
> Exploració bàsica amb una API de laboratori.

[JSONPlaceholder](https://jsonplaceholder.typicode.com/) és una API de proves: accepta peticions de tots els mètodes i respon com si funcionessin, però **no desa cap canvi**.

In [2]:
# URL base de l'API. Tots els recursos (posts, users, todos...) penjen d'aquesta adreça,
# així que la guardem en una variable per no haver-la d'escriure cada vegada.
BASE_URL = "https://jsonplaceholder.typicode.com"

## 1 i 2. GET de `/posts`, `/users` i `/todos`

> **Enunciat**
>
> **1.** Consulta l'API pública JSONPlaceholder utilitzant el mètode GET per obtenir:
> - Llista de publicacions (/posts)
> - Llista d'usuaris (/users)
> - Llista de tasques (/todos)
>
> **2.** Mostra per pantalla:
> - La quantitat total de cada recurs.
> - El codi d'estat de cada petició.

Com que les tres peticions segueixen el mateix patró, les fem dins d'un bucle. Per a cada recurs mostrem el codi d'estat i la quantitat d'elements (la resposta és una llista de diccionaris, així que n'hi ha prou amb `len()`).

In [3]:
# Llista amb els tres recursos que volem consultar
recursos = ["posts", "users", "todos"]

# Diccionari on guardarem les dades de cada recurs (clau = nom del recurs)
dades = {}

# Recorrem els recursos un per un: així fem les tres peticions amb el mateix codi
for recurs in recursos:
    # GET: demanem les dades. La URL final queda, per exemple, .../posts
    # timeout=10 → si el servidor no respon en 10 segons, la petició s'atura (evita esperes infinites)
    resposta = requests.get(f"{BASE_URL}/{recurs}", timeout=10)

    # .json() converteix el text de la resposta en objectes de Python.
    # Aquí obtenim una LLISTA de diccionaris (un diccionari per element).
    dades[recurs] = resposta.json()

    # .status_code → codi d'estat HTTP (200 = tot correcte)
    # len() → nombre d'elements de la llista = quantitat total del recurs
    print(f"/{recurs:<6} → codi d'estat: {resposta.status_code} | total d'elements: {len(dades[recurs])}")

/posts  → codi d'estat: 200 | total d'elements: 100
/users  → codi d'estat: 200 | total d'elements: 10
/todos  → codi d'estat: 200 | total d'elements: 200


Resultat esperat: els tres recursos retornen **200 OK**, amb 100 publicacions, 10 usuaris i 200 tasques.

## 3. Petició a una publicació inexistent (404)

> **Enunciat**
>
> **3.** Fes una petició a una publicació inexistent per obtenir un error 404 i mostra el codi d'estat rebut.

Només hi ha 100 publicacions, així que qualsevol `id` superior no existeix.

In [4]:
# Demanem la publicació amb id 9999, que no existeix (només n'hi ha 100)
resposta_404 = requests.get(f"{BASE_URL}/posts/9999", timeout=10)

# El codi d'estat hauria de ser 404 (Not Found)
print("Codi d'estat:", resposta_404.status_code)

# .ok val True si el codi és menor que 400 (petició correcta) i False si és un error
print("Petició correcta?", resposta_404.ok)

# El cos de la resposta: en aquest cas un diccionari buit, perquè no hi ha cap publicació
print("Contingut:", resposta_404.json())

Codi d'estat: 404
Petició correcta? False
Contingut: {}


El servidor respon **404 Not Found**: el recurs sol·licitat no existeix. És un error de tipus 4xx (error del client), per tant reintentar la mateixa petició no serviria de res.

## 4. POST — crear una publicació fictícia

> **Enunciat**
>
> **4.** Fes una petició POST per crear una nova publicació fictícia. Inclou el títol, el cos del missatge i un userId.
> - Mostra la resposta JSON.
> - Mostra el codi d'estat.

Enviem les dades amb el paràmetre `json=`, que converteix el diccionari a JSON i afegeix la capçalera `Content-Type: application/json`.

In [5]:
# Dades de la nova publicació. Són els camps que demana l'enunciat: títol, cos i userId
nova_publicacio = {
    "title": "La meva primera publicació",
    "body": "Aquest missatge s'ha creat des de Python amb requests.",
    "userId": 1,
}

# POST: enviem dades al servidor per CREAR un recurs nou.
# json=... converteix el diccionari a format JSON i indica al servidor que li enviem JSON
resposta_post = requests.post(f"{BASE_URL}/posts", json=nova_publicacio, timeout=10)

# Esperem 201 (Created): el servidor confirma que ha creat el recurs
print("Codi d'estat:", resposta_post.status_code)

# La resposta retorna la publicació creada, amb l'id que li ha assignat el servidor
print("Resposta JSON:", resposta_post.json())

Codi d'estat: 201
Resposta JSON: {'title': 'La meva primera publicació', 'body': "Aquest missatge s'ha creat des de Python amb requests.", 'userId': 1, 'id': 101}


El codi és **201 Created**: el servidor indica que ha creat un recurs nou i li assigna `id: 101`. Tot i així, com que és una API de proves, la publicació no es desa realment.

## 5. PATCH — modificar parcialment una publicació

> **Enunciat**
>
> **5.** Fes una petició PATCH per modificar parcialment una publicació existent.
> - Mostra la resposta JSON.
> - Mostra el codi d'estat.

Amb PATCH només enviem els camps que volem canviar (en aquest cas, el títol). La resta de camps es mantenen.

In [6]:
# Només indiquem el camp que volem canviar: el títol
canvis = {"title": "Títol modificat amb PATCH"}

# PATCH: MODIFICA PARCIALMENT un recurs existent (aquí, la publicació amb id 1).
# Els camps que no enviem (body, userId) es mantenen com estaven.
resposta_patch = requests.patch(f"{BASE_URL}/posts/1", json=canvis, timeout=10)

# Esperem 200 (OK): la modificació s'ha fet correctament
print("Codi d'estat:", resposta_patch.status_code)

# La resposta mostra la publicació sencera amb el títol ja canviat
print("Resposta JSON:", resposta_patch.json())

Codi d'estat: 200
Resposta JSON: {'userId': 1, 'id': 1, 'title': 'Títol modificat amb PATCH', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


El codi és **200 OK** i la resposta mostra la publicació 1 amb el títol nou i la resta de camps (`body`, `userId`, `id`) intactes. Amb PUT, en canvi, s'hauria de reemplaçar el recurs sencer.

## 6. DELETE — eliminar una publicació

> **Enunciat**
>
> **6.** Fes una petició DELETE sobre una publicació.
> - Mostra la resposta JSON.
> - Mostra el codi d'estat.



In [7]:
# DELETE: ELIMINA un recurs (aquí, la publicació amb id 1). No cal enviar dades.
resposta_delete = requests.delete(f"{BASE_URL}/posts/1", timeout=10)

# Esperem 200 (OK)
print("Codi d'estat:", resposta_delete.status_code)

# La resposta és un diccionari buit {}: el recurs ja no existeix i no hi ha res a retornar
print("Resposta JSON:", resposta_delete.json())

Codi d'estat: 200
Resposta JSON: {}


El codi és **200 OK** i la resposta és un diccionari buit `{}`: no hi ha res a retornar perquè el recurs s'ha eliminat. (Moltes APIs responen en aquest cas amb **204 No Content**.)

### Resum del nivell 1

| Petició | Mètode | Codi d'estat |
|---|---|---|
| `/posts`, `/users`, `/todos` | GET | 200 OK |
| `/posts/9999` | GET | 404 Not Found |
| `/posts` | POST | 201 Created |
| `/posts/1` | PATCH | 200 OK |
| `/posts/1` | DELETE | 200 OK |

---
# Nivell 2 — API pública real: Open-Meteo

> **Enunciat**
>
> Interacció amb una API pública real.
>
> **1.** Explora el repositori de Public APIs i tria una API que permeti fer peticions GET.
>
> **2.** Llegeix la documentació de l'API:
> - Revisa la secció d'endpoints disponibles. En un markdown, apunta com a mínim dos endpoints diferents.
> - Comprova si ofereix filtres o paràmetres opcionals interessants. En un markdown, anota'ls.
> - Verifica que la resposta sigui en format JSON (és el més adequat per a aquest exercici).

## 1. API escollida

Del repositori [Public APIs](https://github.com/public-apis/public-apis), secció **Weather**, he triat **[Open-Meteo](https://open-meteo.com/)**:
- **Auth:** No (no cal clau d'API).
- **HTTPS:** Sí.
- **Format de resposta:** JSON.
- Ofereix previsions meteorològiques gratuïtes per a qualsevol punt del món.

## 2. Documentació

### Endpoints
1. **Previsió meteorològica** — `https://api.open-meteo.com/v1/forecast`
   Retorna la previsió horària o diària per a unes coordenades.
2. **Geocodificació** — `https://geocoding-api.open-meteo.com/v1/search`
   Busca una ciutat pel nom i en retorna les coordenades, el país, la població, etc.
3. *(extra)* **Dades històriques** — `https://archive-api.open-meteo.com/v1/archive`
   Retorna dades meteorològiques de dates passades.

### Paràmetres i filtres interessants
**Endpoint `/v1/forecast`:**
- `latitude`, `longitude` (obligatoris): coordenades del punt.
- `hourly`: variables horàries separades per comes (`temperature_2m`, `relative_humidity_2m`, `precipitation_probability`, `wind_speed_10m`...).
- `daily`: variables diàries (`temperature_2m_max`, `temperature_2m_min`...).
- `timezone`: zona horària de les dates (p. ex. `Europe/Madrid`).
- `forecast_days`: nombre de dies de previsió (1–16).

**Endpoint `/v1/search`:**
- `name` (obligatori): nom de la ciutat.
- `count`: nombre màxim de resultats.
- `language`: idioma dels noms retornats.

### Format
Les dues respostes són **JSON**. Ho comprovem a la capçalera `Content-Type` de la resposta.

## 3. Petició GET senzilla

> **Enunciat**
>
> **3.** Fes una petició GET senzilla:
> - Mostra el codi d'estat de la resposta.
> - Imprimeix de forma clara alguns dels camps de la resposta JSON.

Primer fem servir l'endpoint de **geocodificació** per trobar les coordenades de Barcelona.

In [8]:
# Endpoint de geocodificació: busca una ciutat pel nom i en retorna les coordenades
URL_GEO = "https://geocoding-api.open-meteo.com/v1/search"

# Paràmetres de la consulta. requests els afegeix a la URL automàticament:
# ...?name=Barcelona&count=5&language=ca
params_geo = {
    "name": "Barcelona",  # ciutat que busquem
    "count": 5,           # nombre màxim de resultats
    "language": "ca",     # idioma dels noms retornats
}

# GET amb paràmetres
resposta_geo = requests.get(URL_GEO, params=params_geo, timeout=10)

# Codi d'estat (esperem 200)
print("Codi d'estat:", resposta_geo.status_code)

# La capçalera Content-Type ens confirma que la resposta és JSON
print("Content-Type:", resposta_geo.headers["Content-Type"])

# .url mostra la URL completa que s'ha enviat, amb els paràmetres ja afegits
print("URL final:", resposta_geo.url)

Codi d'estat: 200
Content-Type: application/json; charset=utf-8
URL final: https://geocoding-api.open-meteo.com/v1/search?name=Barcelona&count=5&language=ca


In [9]:
# La resposta és un diccionari; la llista de ciutats trobades és dins de la clau "results"
resultats_geo = resposta_geo.json()["results"]

# Mostrem de forma clara alguns camps de cada resultat
for ciutat in resultats_geo:
    # .get(clau, valor_per_defecte) evita un error si algun resultat no té aquell camp
    print(f"{ciutat['name']} ({ciutat.get('country', '-')}) → "
          f"lat: {ciutat['latitude']}, lon: {ciutat['longitude']}, "
          f"població: {ciutat.get('population', 'desconeguda')}")

Barcelona (Espanya) → lat: 41.38879, lon: 2.15899, població: 1686208
Barcelona (Veneçuela) → lat: 10.1384, lon: -64.68769, població: 815141
Barcelona (Filipines) → lat: 12.8683, lon: 124.1419, població: 3258
Barcelona (Brasil) → lat: -5.95056, lon: -35.92639, població: 3986
Barcelona (Filipines) → lat: 8.15917, lon: 126.43417, població: 3541


Ens quedem amb el primer resultat (Barcelona, Espanya) i en fem servir les coordenades per demanar la previsió.

In [10]:
# Agafem el primer resultat de la cerca (Barcelona, Espanya)
barcelona = resultats_geo[0]

# Endpoint de previsió meteorològica
URL_FORECAST = "https://api.open-meteo.com/v1/forecast"

# Paràmetres: coordenades de Barcelona + variables que volem + opcions
params_forecast = {
    "latitude": barcelona["latitude"],
    "longitude": barcelona["longitude"],
    # variables horàries: temperatura, humitat, probabilitat de pluja i velocitat del vent
    "hourly": "temperature_2m,relative_humidity_2m,precipitation_probability,wind_speed_10m",
    "timezone": "Europe/Madrid",  # hores en horari de Madrid
    "forecast_days": 3,           # previsió per als propers 3 dies
}

resposta_meteo = requests.get(URL_FORECAST, params=params_forecast, timeout=10)

# raise_for_status() atura el programa amb un error si el codi és 4xx o 5xx.
# Així no seguim treballant amb una resposta que ha fallat.
resposta_meteo.raise_for_status()

# Convertim la resposta a diccionari de Python
meteo = resposta_meteo.json()

print("Codi d'estat:", resposta_meteo.status_code)
print("Claus principals del JSON:", list(meteo.keys()))  # per veure com està organitzada la resposta
print("Zona horària:", meteo["timezone"])
print("Unitats:", meteo["hourly_units"])                  # unitat de cada variable (°C, %, km/h...)
# Primer valor horari: primera hora i la seva temperatura amb la unitat
print("Primera hora:", meteo["hourly"]["time"][0],
      "→", meteo["hourly"]["temperature_2m"][0], meteo["hourly_units"]["temperature_2m"])

Codi d'estat: 200
Claus principals del JSON: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
Zona horària: Europe/Madrid
Unitats: {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'precipitation_probability': '%', 'wind_speed_10m': 'km/h'}
Primera hora: 2026-09-28T00:00 → 23.1 °C


## 4. Conversió a DataFrame

> **Enunciat**
>
> **4.** Converteix les dades a un DataFrame de pandas:
> - Mostra les primeres files del DataFrame.

Les dades horàries són a `meteo["hourly"]`: un diccionari on cada clau és una variable i cada valor és una llista (una columna). Aquest format es pot passar directament a `pd.DataFrame`.

In [11]:
# meteo["hourly"] és un diccionari de llistes: cada clau és una columna i cada llista, els seus valors.
# pd.DataFrame() el converteix directament en una taula.
df_meteo = pd.DataFrame(meteo["hourly"])

# La columna "time" arriba com a text; la convertim a tipus data/hora per poder treballar-hi
df_meteo["time"] = pd.to_datetime(df_meteo["time"])

# Mostrem les 5 primeres files
df_meteo.head()

,time,temperature_2m,relative_humidity_2m,precipitation_probability,wind_speed_10m
0,2026-09-28 00:00:00,23.1,80,8,3.7
1,2026-09-28 01:00:00,22.9,79,10,3.6
2,2026-09-28 02:00:00,23.0,78,8,3.3
3,2026-09-28 03:00:00,22.6,81,0,3.6
4,2026-09-28 04:00:00,22.2,85,0,5.1


In [12]:
# .shape → (nombre de files, nombre de columnes)
print("Files i columnes:", df_meteo.shape)

# .describe() → resum estadístic de les columnes numèriques (mitjana, mínim, màxim...)
df_meteo.describe()

Files i columnes: (72, 5)


,time,temperature_2m,relative_humidity_2m,precipitation_probability,wind_speed_10m
count,72,72.000000,72.000000,72.000000,72.000000
mean,2026-09-29 11:30:00,23.112500,82.180556,16.986111,6.155556
min,2026-09-28 00:00:00,20.700000,60.000000,0.000000,2.400000
25%,2026-09-28 17:45:00,21.700000,75.750000,1.750000,4.300000
50%,2026-09-29 11:30:00,22.800000,86.000000,6.500000,5.450000
75%,2026-09-30 05:15:00,24.400000,89.000000,23.500000,8.125000
max,2026-09-30 23:00:00,27.500000,97.000000,83.000000,12.100000
std,NaN,1.802028,10.377829,22.371066,2.438823


Els resultats de la geocodificació (una llista de diccionaris) també es poden convertir directament:

In [13]:
# Els resultats de la geocodificació són una llista de diccionaris:
# cada diccionari es converteix en una fila del DataFrame
df_ciutats = pd.DataFrame(resultats_geo)

# Mostrem només les columnes més interessants
df_ciutats[["name", "country", "admin1", "latitude", "longitude", "population"]].head()

,name,country,admin1,latitude,longitude,population
0,Barcelona,Espanya,Catalunya,41.38879,2.15899,1686208
1,Barcelona,Veneçuela,Estado Anzoátegui,10.13840,-64.68769,815141
2,Barcelona,Filipines,Regió de Bicol,12.86830,124.14190,3258
3,Barcelona,Brasil,Rio Grande do Norte,-5.95056,-35.92639,3986
4,Barcelona,Filipines,Caraga,8.15917,126.43417,3541


---
# Nivell 3 — Open Data BCN

> **Enunciat**
>
> API d'Open Data Barcelona.

L'API d'[Open Data BCN](https://opendata-ajuntament.barcelona.cat/ca/desenvolupadors#APIS) funciona amb **CKAN**. Totes les accions segueixen el patró:

`https://opendata-ajuntament.barcelona.cat/data/api/3/action/<acció>`

Les respostes tenen sempre l'estructura `{"success": ..., "result": ...}`: les dades útils són dins de `result`.

In [14]:
# URL base de l'API d'Open Data BCN (sistema CKAN).
# Afegint-hi el nom de l'acció (package_search, package_show...) obtenim cada endpoint.
BCN_API = "https://opendata-ajuntament.barcelona.cat/data/api/3/action"

## 1a. Cercar datasets amb `package_search`

> **Enunciat**
>
> **1.** Utilitza l'API d'Open Data BCN per:
> - Cercar un dataset del teu interès mitjançant package_search i package_show.

Busquem datasets relacionats amb l'**arbrat** de la ciutat.

In [15]:
# package_search: cerca datasets per paraula clau
resposta_cerca = requests.get(
    f"{BCN_API}/package_search",
    params={"q": "arbrat",  # paraula que busquem
            "rows": 10},    # màxim de datasets a retornar
    timeout=20,
)
resposta_cerca.raise_for_status()  # error si la petició ha fallat

# Les dades útils de CKAN sempre són dins de la clau "result"
cerca = resposta_cerca.json()["result"]

print("Codi d'estat:", resposta_cerca.status_code)
print("Datasets trobats:", cerca["count"])  # total de coincidències
print()

# Llistem el nom tècnic i el títol en català de cada dataset trobat
for dataset in cerca["results"]:
    print(f"- {dataset['name']}")
    print(f"    {dataset['title_translated']['ca']}")

Codi d'estat: 200
Datasets trobats: 7

- atles-biodiversitat-arbrat-rel-geometria-especie
    Relació geometria-espècie de l'arbrat de l'Atles de la Biodiversitat de la ciutat de Barcelona
- atles-biodiversitat-arbrat-geometries
    Inventari de les geometries de l'arbrat de l'Atles de la Biodiversitat de la ciutat de Barcelona
- atles-biodiversitat-arbrat-especies
    Inventari de les espècies d'arbrat de l'Atles de la Biodiversitat de la ciutat de Barcelona
- arbrat-parcs
    Arbrat dels parcs de la ciutat de Barcelona
- arbrat-zona
    Arbrat de zona de la ciutat de Barcelona
- arbres-interes-local
    Arbres d'interès local de la ciutat de Barcelona
- arbrat-viari
    Arbrat viari de la ciutat de Barcelona


## 1b. Detall d'un dataset amb `package_show`

Triem el dataset **`atles-biodiversitat-arbrat-especies`** (inventari de les espècies d'arbres de Barcelona) i en consultem el detall.

In [16]:
# Nom tècnic del dataset escollit (l'hem vist a la cerca anterior)
NOM_DATASET = "atles-biodiversitat-arbrat-especies"

# package_show: retorna tota la informació d'un dataset concret, inclosos els seus recursos (fitxers)
resposta_show = requests.get(f"{BCN_API}/package_show",
                             params={"id": NOM_DATASET}, timeout=20)
resposta_show.raise_for_status()

# Igual que abans, la informació és dins de "result"
dataset = resposta_show.json()["result"]

print("Codi d'estat:", resposta_show.status_code)
print("Títol:", dataset["title_translated"]["ca"])
print("Llicència:", dataset["license_title"])
print("Nombre de recursos:", dataset["num_resources"])

Codi d'estat: 200
Títol: Inventari de les espècies d'arbrat de l'Atles de la Biodiversitat de la ciutat de Barcelona
Llicència: Creative Commons Attribution 4.0
Nombre de recursos: 2


## 2. Selecció d'un recurs en CSV o JSON

> **Enunciat**
>
> **2.** Dels resultats obtinguts, selecciona'n un que tingui recursos disponibles en CSV o JSON.

Llistem els recursos del dataset. Per poder fer servir `datastore_search`, el recurs ha de tenir `datastore_active = True` (vol dir que les dades estan carregades a la base de dades consultable de l'API).

In [17]:
# "resources" és la llista de fitxers del dataset. La passem a DataFrame per veure-la com una taula
# i ens quedem amb les columnes que ens interessen:
#   name → nom del fitxer | format → CSV, JSON... |
#   datastore_active → si es pot consultar amb datastore_search | id → el resource_id
df_recursos = pd.DataFrame(dataset["resources"])[["name", "format", "datastore_active", "id"]]
df_recursos

,name,format,datastore_active,id
0,ab_vw_arbrat_especies.csv,CSV,True,c654c488-2822-4e51-ad9b-f443989a4b41
1,ab_vw_arbrat_especies.json,JSON,False,a182e12d-039b-455a-bccb-ece021d4a98e


In [18]:
# Busquem el primer recurs que compleixi les dues condicions:
#   1) format CSV o JSON (ho demana l'enunciat)
#   2) datastore_active = True (si no, datastore_search no funcionaria)
# next(...) retorna el primer element que compleix la condició
recurs = next(r for r in dataset["resources"]
              if r["format"] in ("CSV", "JSON") and r["datastore_active"])

# Guardem el seu identificador: el necessitem per a la consulta següent
RESOURCE_ID = recurs["id"]

print("Recurs seleccionat:", recurs["name"], f"({recurs['format']})")
print("resource_id:", RESOURCE_ID)

Recurs seleccionat: ab_vw_arbrat_especies.csv (CSV)
resource_id: c654c488-2822-4e51-ad9b-f443989a4b41


## 3. Consulta amb `datastore_search` (mínim 100 registres)

> **Enunciat**
>
> **3.** Amb el resource_id del recurs seleccionat, realitza una consulta mitjançant datastore_search per:
> - Recuperar almenys 100 registres.

Per defecte, `datastore_search` només retorna 100 registres. Amb el paràmetre `limit` en demanem més (aquí, 200).

In [19]:
# datastore_search: retorna els registres (files) d'un recurs
resposta_ds = requests.get(
    f"{BCN_API}/datastore_search",
    params={"resource_id": RESOURCE_ID,  # quin recurs volem
            "limit": 200},               # quants registres (per defecte en dona 100)
    timeout=30,
)
resposta_ds.raise_for_status()

# La resposta és anidada: els registres són a result → records
resultat_ds = resposta_ds.json()["result"]
registres = resultat_ds["records"]   # llista de diccionaris, un per registre

print("Codi d'estat:", resposta_ds.status_code)
print("Registres totals al recurs:", resultat_ds["total"])  # quants n'hi ha en total
print("Registres recuperats:", len(registres))              # quants n'hem descarregat (≥ 100)
# "fields" descriu les columnes; en traiem només el nom de cadascuna
print("Camps:", [camp["id"] for camp in resultat_ds["fields"]])

Codi d'estat: 200
Registres totals al recurs: 525
Registres recuperats: 200
Camps: ['_id', 'especie_id', 'nom_ctf', 'nom_cat', 'nom_cas', 'nom_eng', 'desc_cat', 'desc_cas', 'desc_eng']


## 4. Conversió a DataFrame

> **Enunciat**
>
> **4.** Converteix els resultats a un DataFrame.

In [20]:
# Convertim la llista de registres (diccionaris) en un DataFrame: cada registre és una fila
df_arbres = pd.DataFrame(registres)

# "_id" és un número intern del datastore que no aporta informació: l'eliminem
df_arbres = df_arbres.drop(columns="_id")

# Reordenem les columnes: primer l'identificador i els noms, després les descripcions
df_arbres = df_arbres[["especie_id", "nom_ctf", "nom_cat", "nom_cas", "nom_eng",
                       "desc_cat", "desc_cas", "desc_eng"]]

print("Files i columnes:", df_arbres.shape)
df_arbres.head()

Files i columnes: (200, 8)


,especie_id,nom_ctf,nom_cat,nom_cas,nom_eng,desc_cat,desc_cas,desc_eng
0,158,Populus tremula,"Trémol, trèmol","Alamo temblón, álamo temblón",Aspen,"Arbre caducifoli, de capçada arrodonida, origi...","Árbol caducifolio, de copa redondeada, origina...",This is a deciduous tree with a round crown. I...
1,2437,Laurus nobilis 'Crispa',Llorer,Laurel,Sweet bay,"Arbret o arbust perennifoli, de capçada ovoïda...","Arbolillo o arbusto perennifolio, de copa ovoi...",An evergreen shrub or small tree with an ovoid...
2,840,Acer negundo,Negundo,"Arce negundo, Negundo",Box elder,"Arbre caducifoli, de capçada arrodonida i irre...","Árbol caducifolio, de copa redondeada e irregu...","A deciduous tree with a round, irregular crown..."
3,970,Broussonetia papyrifera,Morera del paper,Morera del papel,Paper mulberry,"És un arbre caducifoli, de capçada arrodonida....","Es un árbol caducifolio, de copa redondeada. O...",This is a deciduous tree with a round crown. I...
4,4943585,Phymosia umbellata,Fimòsia,Fimosia,Mexican Bush Mallow,"Arbust perenne, poc ramificat, originari de Mè...","Arbusto perenne, poco ramificado, originario d...","A sparsely-branched evergreen shrub, native to..."


## 5. Desar el DataFrame en CSV

> **Enunciat**
>
> **5.** Desa el DataFrame en un fitxer .csv.

Fem servir `encoding="utf-8-sig"` perquè els accents (è, ç, l·l...) es vegin bé si s'obre el fitxer amb Excel.

In [21]:
# Nom del fitxer de sortida (es crea a la mateixa carpeta que el notebook)
NOM_FITXER = "arbrat_especies_bcn.csv"

# to_csv desa el DataFrame en CSV:
#   index=False → no desa la columna d'índex de pandas
#   encoding="utf-8-sig" → els accents es veuen bé també a Excel
df_arbres.to_csv(NOM_FITXER, index=False, encoding="utf-8-sig")

# Comprovació: tornem a llegir el fitxer per assegurar-nos que s'ha desat bé
comprovacio = pd.read_csv(NOM_FITXER)
print(f"Fitxer '{NOM_FITXER}' desat amb {len(comprovacio)} files i {comprovacio.shape[1]} columnes.")

Fitxer 'arbrat_especies_bcn.csv' desat amb 200 files i 8 columnes.


### Opcional: recuperar tots els registres amb paginació

El recurs té més registres dels que hem demanat. Amb el paràmetre `offset` podem anar demanant pàgines fins que no en quedin.

In [22]:
tots = []          # aquí anirem acumulant tots els registres
offset = 0         # posició des d'on comencem a demanar (0 = el primer registre)
MIDA_PAGINA = 100  # registres per petició

# Repetim peticions fins que l'API ja no retorni més registres
while True:
    r = requests.get(
        f"{BCN_API}/datastore_search",
        params={"resource_id": RESOURCE_ID,
                "limit": MIDA_PAGINA,  # quants registres per pàgina
                "offset": offset},     # quants registres saltem (els que ja tenim)
        timeout=30,
    )
    r.raise_for_status()
    pagina = r.json()["result"]["records"]

    if not pagina:          # llista buida → ja no queden registres, sortim del bucle
        break

    tots.extend(pagina)     # afegim els registres d'aquesta pàgina a la llista total
    offset += MIDA_PAGINA   # la propera petició començarà 100 registres més endavant

print("Registres totals recuperats:", len(tots))

Registres totals recuperats: 525


---
# Conclusions

**Què he après amb aquesta pràctica:**

- **Mètodes HTTP:** cada mètode té una funció. GET llegeix dades, POST crea un recurs nou, PATCH en modifica una part i DELETE l'elimina. A `requests` cada mètode té la seva funció (`requests.get()`, `requests.post()`...).
- **Codis d'estat:** indiquen com ha anat la petició. He obtingut 200 (correcte), 201 (recurs creat) i 404 (recurs inexistent). Els 4xx són errors de la petició i els 5xx, errors del servidor.
- **Paràmetres:** amb `params=` s'afegeixen filtres a la URL sense haver-la de construir a mà, i amb `json=` s'envien dades al servidor.
- **De JSON a DataFrame:** `.json()` converteix la resposta en objectes de Python i `pd.DataFrame()` els transforma en una taula.

**Diferències entre les tres APIs:**

| | JSONPlaceholder | Open-Meteo | Open Data BCN |
|---|---|---|---|
| Tipus | API de proves | API pública real | API de dades obertes (CKAN) |
| Autenticació | No | No | No |
| Estructura de la resposta | Llista de diccionaris, directa | Diccionari amb les dades dins de `hourly` o `results` | Molt anidada: `result` → `records` |
| Desa els canvis? | No, només els simula | Només lectura | Només lectura |

**Dificultats i aprenentatges pràctics:**

- Cal **explorar l'estructura del JSON** (per exemple, amb `.keys()`) abans de convertir-lo a DataFrame, perquè cada API organitza les dades d'una manera diferent.
- A Open Data BCN no tots els recursos es poden consultar amb `datastore_search`: cal comprovar que `datastore_active` sigui `True`.
- Per defecte `datastore_search` només retorna 100 registres. Per obtenir-ne més cal fer servir `limit` i, per descarregar-ho tot, paginar amb `offset`.
- És recomanable posar sempre `timeout` i comprovar el codi d'estat (`raise_for_status()`) per no treballar amb respostes que han fallat.